# Set up Wednesday's return files (week 3, day 3)

For the trainer. Run this notebook once in the class workspace before Wednesday, from your Git folder of the course: `week-03/setup/day-03-setup.ipynb`. Run it after Monday's setup notebook, which makes the catalog's schema `returns` and gives the class read access to it. The numbered steps, and what to check, are in `day-03-setup.md` beside it.

It makes the volume `course.returns.raw` and writes the course's return files into it: 52 XML files, one return each, tax years 2020 to 2023. The returns are in `day-03-course-data.py` beside this notebook. The course made up every file: no number, name or amount belongs to a real taxpayer. It also makes one schema for each learner, `course.learner_NN`, where that learner can make tables. Running it again is safe: it writes the same files again, and leaves the learners' tables alone.

Each code cell has a number on its first line. Run cells 1 to 4 in order.

In [ ]:
# 1. Settings. CLASS_GROUP holds the learners. LEARNERS maps each learner's number to the
#    learner's Databricks user name (their sign-in email). Fill it in here in the workspace, and
#    do not commit it: the course repository holds no learner's name or email.
CLASS_GROUP = "account users"
LEARNERS = {
    "00": "",  # the trainer's own schema, for the answers and the demos: your user name
    # "01": "<learner's user name>",
}

import os
try:
    _path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
    _folder = os.path.dirname(_path if _path.startswith("/Workspace") else "/Workspace" + _path)
except NameError:  # outside Databricks: the folder the notebook runs in
    _folder = os.getcwd()
course = {"__name__": "course_data"}
exec(open(os.path.join(_folder, "day-03-course-data.py"), encoding="utf-8").read(), course)
print(len(course["returns"]()), "returns to write;", len([k for k, v in LEARNERS.items() if v]), "learner schemas to make.")

In [ ]:
# 2. The volume, and the 52 return files in it. Files the course did not write are left alone.
for statement in course["volume_statements"](CLASS_GROUP):
    spark.sql(statement)
volume = "/Volumes/course/returns/raw/"
for record in course["returns"]():
    with open(volume + course["file_name"](record), "w", encoding="utf-8") as out:
        out.write(course["xml_text"](record))
print("Wrote", len(os.listdir(volume)), "files to", volume, "and granted READ VOLUME to", CLASS_GROUP)

In [ ]:
# 3. Read the files back as the learners will. Expected: 52 files and 52 rows; 77 dependents and
#    62 W-2s as arrays; F-30037 with 4 returns, F-30074 with 1; F-30111-2021 with adjusted gross
#    income 0; 11 tax-change cases; IRSW2 and DependentDetail each an array.
from pyspark.sql import functions as F
facts = course["setup_facts"]()
returns = (spark.read.format("xml").option("rowTag", "Return").load(volume)
           .select("*", "_metadata.file_name"))
print(len(os.listdir(volume)), "files,", returns.count(), "rows; expected", facts["files"])
print("dependents:", returns.select(F.explode("ReturnData.IRS1040.DependentDetail")).count(), "expected", facts["dependents"])
print("W-2s:", returns.select(F.explode("ReturnData.IRSW2")).count(), "expected", facts["w2s"])
print("tax-change cases:", returns.filter("CourseLabel.ExamChangedTax = 'yes'").count(), "expected", facts["tax_change_cases"])
print("F-30037:", returns.filter("ReturnHeader.Filer.CaseKey = 'F-30037'").count(), "returns; F-30074:",
      returns.filter("ReturnHeader.Filer.CaseKey = 'F-30074'").count())
print("adjusted gross income 0:", [r.file_name for r in returns.filter("ReturnData.IRS1040.AdjustedGrossIncomeAmt = 0").collect()])
data = returns.schema["ReturnData"].dataType
print("IRSW2 is", data["IRSW2"].dataType.typeName(), "| DependentDetail is",
      data["IRS1040"].dataType["DependentDetail"].dataType.typeName())

In [ ]:
# 4. One schema for each learner, course.learner_NN, with the rights to make tables in it.
#    A learner owns the tables they make, so they can replace them. Skips an empty user name.
for number, principal in LEARNERS.items():
    if not principal:
        continue
    for statement in course["learner_statements"](number, principal):
        spark.sql(statement)
    print(f"course.learner_{number}: USE SCHEMA and CREATE TABLE granted")